# 03 — Multiheaded training

This notebook focuses on multiheaded training. This is the key strategy that from the original MACE paper that was recommended for better generalisabiity, and the one that we think might have the most potential for TL. 

Multiheaded training in general involves training with two forms of data simultaneously. For this project typically a low and high fidelity data. The key changes will therefore be how we specify the dataloaders and how to specify the strategy in the same way.

## 1. Base setup

Load a full base model saved from prior training, then set up the data, loss and Trainer as in [tutorial 01](01_base_model_training.ipynb). 

In [ ]:
# Imports
import json
import random
from pathlib import Path

import numpy as np
import torch
import ase.io

from mace.data.atom_data_loader import AtomDataLoaderBuilder
from mace.modules.loss import InvariantsWeightedEnergyForcesNacsDipoleLoss
from mace.training.trainer import Trainer
from mace.training import MultiHeadStrategy, MultiHeadCorrectionStrategy, LoRAStrategy
from mace.testing import Tester

# Set up
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.set_default_dtype(torch.float64)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the full base model and keep it on CPU
base_model_path = "<insert full model file path>"
base_model = torch.load(base_model_path, map_location="cpu", weights_only=False)
base_model = base_model.to(dtype=torch.float64)

# Load LF and HF training and validation data
lf_train_atoms = ase.io.read("<insert LF training file path>", index=":")
hf_train_atoms = ase.io.read("<insert HF training file path>", index=":")
lf_valid_atoms = ase.io.read("<insert LF validation file path>", index=":")
hf_valid_atoms = ase.io.read("<insert HF validation file path>", index=":")

# Same loss and Trainer setup as before
loss_fn = InvariantsWeightedEnergyForcesNacsDipoleLoss(
    energy_weight=1.0, forces_weight=1.0, nacs_weight=1.0,
).to(device)
trainer = Trainer(
    device=device, max_epochs=100, optimiser_lr=1e-3,
    ema_decay=0.99, restore_best=True,
)

# Create a new directory for this run
output_dir = Path("<insert new multi-head run directory>")
output_dir.mkdir(parents=True, exist_ok=False)

## 2. Prepare multi-head data and loaders

### Specifying Labels

When doing multihead training, we need to include more than one data type. These will have to be specified accordingly with consistent labels. 

For the training and data validation sets for example, instead of parsing the databuilder a list of ASE atoms, we parse in a dictionary with keys as labels and values being lists of ASE atoms. 
In the example below, the labels used are CASSCF and NEVPT2, representing LF and HF data respectively. 

Different levels of theory will also yield different E0s. Therefore, we will similarly input E0s as a dictionary of dictionaries, with the keys being the matching labels. As long as the labels names match, everything else has the same syntax and will be handeled internally.

One key difference when using the data loader builders is that we should now specify an argument `balance_heads`. In multihead training, the dataset sizes will often be unequal. For example, LF data is usually much more abundunt than HF data, so the LF data might have 10x or 100x more ASE Atoms. If trained normally, this would likely lead to very biased and unsuccessful tuning (especially if the goal is to learn sparse amounts of data). As such, balance heads should be set to True especially when the datasets are very unbalanced. 

During the various epochs, the dataset from the more abundunt source (in this case LF data) will be randomly chosen such that every batch of HF data will have an equal amount of LF data. This ensures that the full LF data will be "seen" but the training process will not be too skewed toward the more abundant datasource. 

In [ ]:
train_data = {"CASSCF": lf_train_atoms, "NEVPT2": hf_train_atoms}
valid_data = {"CASSCF": lf_valid_atoms, "NEVPT2": hf_valid_atoms}

# One element-to-E0 dictionary for each fidelity
e0s_by_head = {
    "CASSCF": {"H": -15.357929595328724, "C": -1032.083979117871},
    "NEVPT2": {"H": -15.712048126191444, "C": -1035.5115207879423},
}

# Create Data Loader Builder, same as before, but now E0s is a dict of dicts
multihead_builder = AtomDataLoaderBuilder(
    cutoff=float(base_model.r_max), 
    E0s=e0s_by_head
)

# Set up train loaders similar to before, but now with the addition of balance heads
train_loader = multihead_builder.load(
    train_data, 
    batch_size=10,
    shuffle=True, 
    seed=seed, 
    balance_heads=True
)

valid_loader = multihead_builder.load(
    valid_data, 
    batch_size=10, 
    shuffle=False, 
    balance_heads=False
)
metadata = multihead_builder.get_metadata()

## 3. Multi-head strategy


`MultiHeadStrategy` copies the base model and duplicates its autoencoder head for each dataset. This head refers to everything that isn't the GNN. The HF data will learn new readouts whilst keeping the graph layer shared.
This prevents the limited HF data from modifying or changing the GNN outputs too much (in theory) since they are "constrained" by the LF data.

### Inputs and defaults

- `metadata` supplies the head names, head count and E0s from the builder.
- `gnn_lr=0.01` is the default learning-rate multiplier for the shared backbone.
- `head_multipliers` is a dictionary of learning-rate multipliers whose keys must match the metadata's head names. If omitted, the first head uses `0.01` and the remaining heads use `1.0`. This is important because the various heads realistically should not train and the same LR. Typically the initial LF data has already be trained and learned, so its head should recieve a lower learning rate, while the new HF data would require a larger LR. 
These values specified in the multiplier and relative, and multiply the Trainer's learning rate For example, `1e-3 × 0.01 = 1e-5`.

A head multiplier of zero freezes that head. Its predictions can still change if the shared backbone is updated.

In [ ]:
# Define the strategy and learning-rate multipliers
strategy = MultiHeadStrategy(
    metadata=metadata, 
    gnn_lr=0.01,
    head_multipliers={"CASSCF": 0.01, "NEVPT2": 1.0}
)

# Apply to a copy of the original single-head model
multihead_model = strategy.apply(base_model)

## 4. Combine multi-head training with LoRA

Strategies can also be applied in sequence, as in [tutorial 02](02_transfer_learning_strategies.ipynb). 

Below we show how to use both multihead and LoRa together to output a new model

In [ ]:
# The multi-head strategy has already been applied above
# Find the HF head, then add LoRA to its decoder only
hf_index = metadata.head_to_index["NEVPT2"]

# Inject lora to the transfer model 
hf_lora_strategy = LoRAStrategy(
    rank=4, 
    alpha=1.0,
    lora_layers=[f"autoencoder_heads.{hf_index}.perm_decoder"]
)

lora_multihead_model = hf_lora_strategy.apply(multihead_model)

## 5. Correction-head strategy

Thus far, the multihead training proposed uses independent heads from each other. Meaning each fidelity level of data has one independent head. Another alternative we conceptualised is instead to use a multihead-correction strategy. In this strategy, the HF data learns a correction instead of independent predictions. 

```text
LF data → shared backbone → base head
HF data → shared backbone → base head + correction head
```

The key motivation behind this is that if the LF data can learn most of the PES, then when using small amounts of HF data, we should not forget whatever the LF PES has mapped out. So the key assumption is that the "correction" surface is relatively easier to learn. 

It is also partly inspired by the fact that CASPT2 or perturvative methods are usually a correction of the CASSCF, so this architecture tries to model that behaviour.

Importantly, 
1) For the correction heads to work, we have to use direct predictions. This is because the correction surfaces do not correlate with each other the same way the ES surfaces correlate. As such, diagonalising a matrix will yield spectacularly poor results. This is handelled internally.
2) The errors in predictions seems to be small relative to the variations in the correction surfaces, so its not too clear if this correction is actually effective yet (though brief ablation studies seem to suggest that this slightly outperforms independent heads).
3) One advantage of this correcion head is that it might prevent catastrophic errors in the order of thousands of eV (which we do see happen when testing on dynamic data). This is because the LF data might help form a more stable base that doesn't "explode" at very distorted geometries. 

In terms of usage, the xyz file now expects a label `REF_raw_energy_differences`. This is the raw difference in energies between the HF and LF data. These are required for the energy autoencoder because the targets require that information. 

The data loader builder should be specified with an argument `energy_difference_references`. This expects a dictionary, and is required to prepare the centered correction targets fro the encoder.

The rest of the training and strategy then follow the same syntax as the above multihead training.  

In [ ]:
# HF train/validation labels must include REF_raw_energy_differences:
# NEVPT2 minus CASSCF energies for the same geometry, shape [1, states].
correction_builder = AtomDataLoaderBuilder(
    cutoff=float(base_model.r_max), 
    E0s=e0s_by_head,
    energy_difference_references={"CASSCF": None, "NEVPT2": "CASSCF"},
)
correction_train_loader = correction_builder.load(
    train_data, 
    batch_size=10, 
    shuffle=True, 
    seed=seed, 
    balance_heads=True,
)
correction_valid_loader = correction_builder.load(
    valid_data, 
    batch_size=10, 
    shuffle=False, 
    balance_heads=False,
)
correction_metadata = correction_builder.get_metadata()

# Apply to the original single-head model (currently requires 3 states and latent_dim=16)
correction_strategy = MultiHeadCorrectionStrategy(
    metadata=correction_metadata, 
    gnn_lr=0.01,
    head_multipliers={"CASSCF": 0.01, "NEVPT2": 1.0},
)

correction_model = correction_strategy.apply(base_model)
